#### Etape 2.2 : Fusion et enrichissement
- Charger les consommations nettoyees (depuis Parquet)
- Fusionner avec les donnees meteo (sur commune et timestamp arrondi a l'heure)
- Fusionner avec le referentiel batiments
- Fusionner avec les tarifs pour calculer le cout financier
- Creer des features derivees :
  - Consommation par occupant
  - Consommation par m2
  - Cout journalier, mensuel, annuel
  - Indice de performance energetique (IPE)
  - Ecart a la moyenne de la categorie

In [7]:
import os
import pandas as pd

DATA_DIR = "../data" #WHAT ?
DATA_DIR = os.path.join(DATA_DIR, "..", "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "..", "output", "consommation_clean")
OUTPUT_METEO = os.path.join(DATA_DIR, "..", "output", "meteo_clean.csv")
DATA_TARIF = os.path.join(DATA_DIR, "..", "data", "tarifs_energie.csv")

OUTPUT_FULL_DF = os.path.join(DATA_DIR, "..", "output", "consommations_enrichies.csv")
OUTPUT_FULL_DF_PARQUET = os.path.join(DATA_DIR, "..", "output", "consommations_enrichies.parquet")




df_conso = pd.read_parquet(OUTPUT_DIR)
df_meteo = pd.read_csv(OUTPUT_METEO)
df_tarif = pd.read_csv(DATA_TARIF)

print(df_conso.dtypes)
print(len(df_conso))
print()
print(df_meteo.dtypes)
print()
print(df_tarif.dtypes)


batiment_id                str
unite                      str
conso_clean            float64
hour                     int32
year                     int32
month                    int32
nom                        str
type                       str
commune                    str
surface_m2               int32
annee_construction       int32
classe_energetique         str
nb_occupants_moyen       int32
date                  category
type_energie          category
dtype: object
7492584

commune                        str
timestamp                      str
temperature_c              float64
humidite_pct               float64
rayonnement_solaire_wm2    float64
vitesse_vent_kmh           float64
precipitation_mm           float64
date                           str
day                          int64
day_of_week                  int64
month                        int64
season                         str
dtype: object

date_debut            str
date_fin              str
type_energie          str

- Fusionner avec les donnees meteo (sur commune et timestamp arrondi a l'heure)

In [8]:

df_fusion_meteo = df_conso \
    .merge(
        df_meteo, on = ['date', 'hour', 'commune', 'month'],
        how = "left"
    )

print(df_fusion_meteo)

KeyError: 'hour'

In [ ]:
# df_fusion_metro_tarif = df_fusion_meteo.merge(
#     df_tarif, on="type_energie",
#      how="left"
#      ).query("date_debut <= date <= date_fin") \
#     .drop(columns=["date_debut", "date_fin"])
#marche pas, c'est cartésien

#on casse le df des tarifs

def apply_tarif(df, df_tarif):
    df = df.copy()

    for energie in df_tarif["type_energie"].unique():
        mask_energie = df["type_energie"] == energie
        tarifs = df_tarif[df_tarif["type_energie"] == energie]

        for _, r in tarifs.iterrows():
            mask_date = (df["date"] >= r["date_debut"]) & (df["date"] <= r["date_fin"])
            df.loc[mask_energie & mask_date, "tarif_unitaire"] = r["tarif_unitaire"]

    return df

df_fusion_metro_tarif = apply_tarif(df_fusion_meteo, df_tarif)


- Creer des features derivees :
  - Consommation par occupant
  - Consommation par m2
  - Cout journalier, mensuel, annuel
  - Indice de performance energetique (IPE)
  - Ecart a la moyenne de la categorie

In [ ]:

df_stats = df_fusion_metro_tarif[[
    "batiment_id",
    "type",
    "commune",
    "type_energie",
    "conso_clean",
    "unite",
    "date",
    "hour",
    "year",
    "month",
    "day_of_week",
    "surface_m2",
    "nb_occupants_moyen",
    "tarif_unitaire",
    "classe_energetique",
]].copy()

df_stats["conso_annuelle"] = df_stats.groupby(["batiment_id", "year"])["conso_clean"].transform("sum")

df_stats["conso_moyenne_par_occupant_annee"] = df_stats["conso_annuelle"] / df_stats["nb_occupants_moyen"]

df_stats["conso_m2"] = df_stats["conso_clean"] / df_stats["surface_m2"]

df_stats["conso_m2_annee"] = df_stats["conso_annuelle"] / df_stats["surface_m2"]

df_stats["cout_horaire"] = df_stats["conso_clean"] * df_stats["tarif_unitaire"]

df_stats["cout_juor"] = df_stats.groupby(["batiment_id", "date"])["cout_horaire"].transform("sum")

df_stats["cout_mois"] = df_stats.groupby(["batiment_id", "month"])["cout_horaire"].transform("sum")

df_stats["cout_annee"] = df_stats.groupby(["batiment_id", "year"])["cout_horaire"].transform("sum")

df_stats["ipe"] = df_stats["conso_annuelle"] / df_stats["surface_m2"]

df_stats["ipe_moyenne_categorie"] = df_stats.groupby("type")["ipe"].transform("mean")

df_stats["ecart_ipe"] = ((df_stats["ipe"] - df_stats["ipe_moyenne_categorie"]) / df_stats["ipe_moyenne_categorie"]) * 100


df_stats = df_stats.drop(
                columns=[
                "day_of_week",
                "surface_m2",
                "nb_occupants_moyen",
                "tarif_unitaire",
                "ipe_moyenne_categorie"
            ]
)


print(df_stats.loc[df_stats["batiment_id"] == "BAT0001"])



KeyError: "['day'] not in index"

#### Export csv parquet

In [ ]:

df_stats.to_csv(OUTPUT_FULL_DF)

df_stats.to_parquet(
    OUTPUT_FULL_DF_PARQUET,
    engine="pyarrow",
    index=False
)